# Sketch 1.2 - standalone, portable notebook

Self-contained extract of **sketch_1_2_leaderboard** from the combined *9 sketches*
notebook. The sketch code is unchanged; the shared setup it needs
(imports, data loading) is included below. Chart data is **inlined** (no external
`altair-data` files) so it renders on Linux, macOS and Windows alike.

## Shared setup

In [1]:
import altair as alt
import pandas as pd
import numpy as np
import geopandas as gpd  # conda install -c conda-forge geopandas
import json

# Inline the chart data so every HTML output is SELF-CONTAINED and renders on
# all platforms (incl. macOS): external altair-data-*.json URL files are not
# fetched by many notebook frontends, which left the charts blank on Mac. The
# search sketches (1.13 / 2.4 / 2.6) restrict their name universe below so the
# inlined tables stay small.
alt.data_transformers.enable('default')
alt.data_transformers.disable_max_rows()
pass

In [2]:
# Memory-lean load: another heavy job is running on this machine (~2.5 GB free),
# so we read the CSV with categorical dtypes (15k name strings stored once), derive
# the numeric year from the 122 'annais' categories instead of 3.5M strings, then
# relax categories back to plain objects (the strings stay shared) so every later
# groupby keeps its normal semantics.
raw = pd.read_csv('dpt2020.csv', sep=';',
                  dtype={'sexe': 'int8', 'preusuel': 'category',
                         'annais': 'category', 'dpt': 'category', 'nombre': 'int32'})
cat_years = pd.to_numeric(pd.Index(raw['annais'].cat.categories), errors='coerce').astype('float32')
raw['year'] = np.asarray(cat_years)[raw['annais'].cat.codes]
raw.drop(columns=['annais'], inplace=True)
raw['decade'] = (raw['year'] // 10 * 10)
raw['preusuel'] = raw['preusuel'].astype(object)  # shared strings, plain groupbys
raw['dpt'] = raw['dpt'].astype(object)

records = raw[raw.dpt != 'XX'].copy()
named = records[records.preusuel != '_PRENOMS_RARES'].copy()

dept_total = records.groupby('dpt', as_index=False)['nombre'].sum().rename(
    columns={'nombre': 'dept_total'})

ALL_NAMES = sorted(named.preusuel.unique().tolist())  # full dropdown option list
DECADES = list(range(1900, 2021, 10))
# Shared period options for charts that need an all-years aggregate (2.4).
DECADE_OPTS = [0] + DECADES
DECADE_LABELS = ['All years'] + [str(d) for d in DECADES]
SEX_OPTIONS = ['Mixed', 'Boys', 'Girls']
SEX_LABELS = ['mixed', 'boys', 'girls']

import gc
del raw
gc.collect()

print(f'{len(named):,} named rows; {len(ALL_NAMES):,} distinct names.')
named.sample(3)


3,668,274 named rows; 15,270 distinct names.


,sexe,preusuel,dpt,nombre,year,decade
374906,1,DENIS,973,4,1981.0,1980.0
1594737,1,TIAGO,30,11,2020.0,2020.0
854178,1,JEROME,50,161,1975.0,1970.0


## The sketch

### Sketch 1.2 - Bump-chart leaderboard (decade-window gauges)

Per-decade rank (1 = top); only names that ever reached the top 3 are coloured and
labelled, the rest are grey context. Two sliders bound the range of decades
shown; hover to highlight.


In [3]:
_dec_base = named[named.year.isin(DECADES)]
dec_mixte = (_dec_base.groupby(['year', 'preusuel'], as_index=False)['nombre'].sum())
dec_mixte['sex_mode'] = 'Mixed'
dec_sex = (_dec_base.groupby(['year', 'preusuel', 'sexe'], as_index=False)['nombre'].sum())
dec_sex['sex_mode'] = np.where(dec_sex['sexe'].eq(1), 'Boys', 'Girls')
dec = pd.concat([
    dec_mixte[['year', 'preusuel', 'nombre', 'sex_mode']],
    dec_sex[['year', 'preusuel', 'nombre', 'sex_mode']],
], ignore_index=True)
dec['rank'] = dec.groupby(['year', 'sex_mode'])['nombre'].rank(ascending=False, method='first')
dec = dec[dec['rank'] <= 40].sort_values(['sex_mode', 'preusuel', 'year']).copy()
dec['run'] = dec.groupby(['sex_mode', 'preusuel'])['year'].transform(lambda s: (s.diff() > 10).cumsum())
# Single-point line groups trigger tiny-skia path-stroking warnings in PNG export.
dec['run_len'] = dec.groupby(['sex_mode', 'preusuel', 'run'])['year'].transform('size')
featured = set(dec.loc[dec['rank'] <= 3, 'preusuel'])
dec['featured'] = dec['preusuel'].isin(featured)
feat, ctx = dec[dec.featured], dec[~dec.featured]
# Anchor each label at the name's last appearance IN THE TOP-8 for the current sex mode.
feat8 = feat[feat['rank'] <= 8]
last_seen = feat8.loc[feat8.groupby(['sex_mode', 'preusuel'])['year'].idxmax()].sort_values(['sex_mode', 'rank', 'year']).copy()
last_seen['k'] = last_seen.groupby(['sex_mode', 'rank']).cumcount()
last_seen['rank_lab'] = last_seen['rank'] + (last_seen['k'] % 2) * 0.34 - 0.17

d_from = alt.param(name='d_from', value=1900,
                   bind=alt.binding_range(min=1900, max=2020, step=10, name='From decade  '))
d_to = alt.param(name='d_to', value=2020,
                 bind=alt.binding_range(min=1900, max=2020, step=10, name='To decade  '))
sex_12 = alt.param(name='sex_12', value='Mixed',
                   bind=alt.binding_select(options=SEX_OPTIONS, labels=SEX_LABELS,
                                           name='Sex  '))
in_window = 'datum.sex_mode == sex_12 && datum.year >= d_from && datum.year <= d_to && datum.rank <= k_rank'

k_rank = alt.param(name='k_rank', value=8,
                   bind=alt.binding_range(min=3, max=40, step=1, name='Ranks shown (top K)  '))
hover = alt.selection_point(fields=['preusuel'], on='pointerover', empty=False)
# Auto-fitting reversed scale: the y domain follows the chosen K (1 stays on top).
y_enc = alt.Y('rank:Q', title='Rank (1 = most common)',
              scale=alt.Scale(reverse=True, zero=False, nice=False, padding=10),
              axis=alt.Axis(values=list(range(1, 41))))
ctx_lines = alt.Chart(ctx).transform_filter(in_window).transform_filter('datum.run_len >= 2').mark_line(
    color='#dcdcdc', strokeWidth=1.1).encode(
    x=alt.X('year:O', title='Decade'), y=y_enc, detail=['sex_mode:N', 'preusuel:N', 'run:N'])
feat_lines = alt.Chart(feat).transform_filter(in_window).transform_filter('datum.run_len >= 2').mark_line(
    point=True, strokeWidth=2.6).encode(
    x=alt.X('year:O', title='Decade'), y=y_enc,
    color=alt.Color('preusuel:N', legend=None, scale=alt.Scale(scheme='category20')),
    detail=['sex_mode:N', 'preusuel:N', 'run:N'],
    opacity=alt.condition(hover, alt.value(1), alt.value(0.9)),
    tooltip=[alt.Tooltip('preusuel:N', title='Name'), alt.Tooltip('sex_mode:N', title='Sex mode'),
             alt.Tooltip('year:O', title='Decade'), alt.Tooltip('rank:Q', title='Rank'),
             alt.Tooltip('nombre:Q', title='Births', format=',')])
feat_labels = alt.Chart(last_seen).transform_filter(in_window).mark_text(
    align='left', dx=8, fontSize=9, fontWeight='bold').encode(
    x='year:O', y='rank_lab:Q', text='preusuel:N',
    color=alt.Color('preusuel:N', legend=None, scale=alt.Scale(scheme='category20')))
sketch_1_2 = (ctx_lines + feat_lines + feat_labels).add_params(hover, d_from, d_to, k_rank, sex_12).properties(
    width=760, height=430,
    title='1.2 - Leaderboard by decade - names that ever reached the top 3 (choose sex)')

sketch_1_2.save('sketch_1_2_leaderboard.png', ppi=120)
sketch_1_2


alt.LayerChart(...)